# E-commerce Data Cleaning — Python Workflow

Notebook này mô tả quy trình xử lý dữ liệu thương mại điện tử bằng Python:
đọc dữ liệu nguồn → kiểm tra cấu trúc → đánh giá chất lượng dữ liệu
→ làm sạch → kiểm tra sau làm sạch → xuất dữ liệu CSV.

Mục tiêu: tạo bộ dữ liệu sạch, nhất quán và sẵn sàng cho phân tích
Data Analyst, xây dựng KPI và trực quan hóa trên Power BI hoặc Tableau.

## 1. Import Libraries and Configure Paths

Import các thư viện cần thiết và thiết lập đường dẫn dữ liệu đầu vào, đầu ra.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Đường dẫn dữ liệu đầu vào
INPUT_FILE = Path(
    r"E:\FPT Polytechnic\DATN1\Data\ecommerce_orders_dataset (1).csv"
)

# Thư mục lưu kết quả
OUTPUT_DIR = INPUT_FILE.parent / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Kiểm tra file nguồn
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Không tìm thấy file dữ liệu: {INPUT_FILE}"
    )

print("Đường dẫn dữ liệu:", INPUT_FILE)
print("Thư mục đầu ra:", OUTPUT_DIR)

Đường dẫn dữ liệu: E:\FPT Polytechnic\DATN1\Data\ecommerce_orders_dataset (1).csv
Thư mục đầu ra: E:\FPT Polytechnic\DATN1\Data\output


Bước 2 - Đọc dữ liệu CSV
## 2. Load Raw Data

Đọc dữ liệu đơn hàng thương mại điện tử từ file CSV.
Kiểm tra kích thước, cấu trúc và các bản ghi đầu tiên.

In [2]:
df_raw = pd.read_csv(INPUT_FILE, low_memory=False)

print("Số dòng ban đầu:", df_raw.shape[0])
print("Số cột ban đầu:", df_raw.shape[1])

display(df_raw.head())
display(df_raw.dtypes.to_frame("Data Type"))

Số dòng ban đầu: 30000
Số cột ban đầu: 41


,Order_ID,Customer_ID,Order_Date,Year,Month,Day,Day_Of_Week,Quarter,Customer_Age,Customer_Gender,...,Delivery_Days,Order_Status,Returned,Review_Rating,Customer_Lifetime_Value,Profit_Margin_Percent,Profit_Amount,Season,Holiday_Season,High_Value_Order
0,615717,CUST007322,2023-01-01,2023,1,1,Sunday,1,32,Male,...,2,Delivered,No,4.4,2144.92,23.10,15.75,Winter,No,No
1,626919,CUST004717,2023-01-01,2023,1,1,Sunday,1,50,Male,...,9,Delivered,No,4.1,817.17,8.57,3.32,Winter,No,No
2,615781,CUST004415,2023-01-01,2023,1,1,Sunday,1,61,Male,...,2,Delivered,No,5.0,541.16,29.72,45.67,Winter,No,No
3,621747,CUST004114,2023-01-01,2023,1,1,Sunday,1,34,Male,...,6,Returned,Yes,3.4,700.49,25.22,75.25,Winter,No,No
4,625881,CUST000145,2023-01-01,2023,1,1,Sunday,1,37,Male,...,8,Delivered,No,3.6,2133.77,24.64,26.06,Winter,No,No


,Data Type
Order_ID,int64
Customer_ID,object
Order_Date,object
Year,int64
Month,int64
Day,int64
Day_Of_Week,object
Quarter,int64
Customer_Age,int64
Customer_Gender,object


Bước 3: Kiểm tra chất lượng dữ liệu
## 3. Data Quality Assessment

Đánh giá chất lượng dữ liệu trước khi làm sạch:
- Kiểm tra giá trị thiếu.
- Kiểm tra bản ghi trùng lặp.
- Thống kê kiểu dữ liệu.
- Kiểm tra phân phối và giá trị bất thường.

In [3]:
missing_report = pd.DataFrame({
    "Missing Count": df_raw.isna().sum(),
    "Missing Percent": (
        df_raw.isna().mean() * 100
    ).round(2),
    "Unique Values": df_raw.nunique(dropna=True),
    "Data Type": df_raw.dtypes.astype(str)
})

print("Số bản ghi trùng hoàn toàn:", df_raw.duplicated().sum())
display(
    missing_report.sort_values(
        "Missing Percent", ascending=False
    )
)
display(df_raw.describe(include="all").T)

Số bản ghi trùng hoàn toàn: 0


,Missing Count,Missing Percent,Unique Values,Data Type
Order_ID,0,0.0,30000,int64
Customer_ID,0,0.0,8683,object
Order_Date,0,0.0,1461,object
Year,0,0.0,4,int64
Month,0,0.0,12,int64
Day,0,0.0,31,int64
Day_Of_Week,0,0.0,7,object
Quarter,0,0.0,4,int64
Customer_Age,0,0.0,58,int64
Customer_Gender,0,0.0,3,object


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Order_ID,30000.0,NaN,NaN,NaN,615000.5,8660.398374,600001.0,607500.75,615000.5,622500.25,630000.0
Customer_ID,30000,8683,CUST000031,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Order_Date,30000,1461,2024-03-09,39,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Year,30000.0,NaN,NaN,NaN,2024.4947,1.118815,2023.0,2023.0,2024.0,2025.0,2026.0
Month,30000.0,NaN,NaN,NaN,6.5315,3.460955,1.0,4.0,7.0,10.0,12.0
Day,30000.0,NaN,NaN,NaN,15.7236,8.779714,1.0,8.0,16.0,23.0,31.0
Day_Of_Week,30000,7,Monday,4327,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quarter,30000.0,NaN,NaN,NaN,2.507,1.118806,1.0,2.0,3.0,4.0,4.0
Customer_Age,30000.0,NaN,NaN,NaN,36.3703,11.315534,18.0,28.0,36.0,44.0,75.0
Customer_Gender,30000,3,Female,14863,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Bước 4: Làm sạch dữ liệu
## 4. Data Cleaning

Thực hiện làm sạch dữ liệu:
- Chuẩn hóa tên cột.
- Loại bỏ khoảng trắng thừa trong dữ liệu văn bản.
- Chuẩn hóa các ô trống thành giá trị thiếu.
- Xóa các bản ghi trùng hoàn toàn.
- Giữ lại các trường có dữ liệu thiếu để đánh giá trước khi quyết định cách xử lý.

In [4]:
df = df_raw.copy()

# Chuẩn hóa tên cột
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

# Làm sạch dữ liệu văn bản
text_columns = df.select_dtypes(
    include=["object", "string"]
).columns

for col in text_columns:
    df[col] = df[col].map(
        lambda x: x.strip() if isinstance(x, str) else x
    )

# Chuẩn hóa chuỗi rỗng và các dạng biểu diễn thiếu phổ biến
missing_tokens = ["", "NA", "N/A", "NULL", "null", "None"]

for col in text_columns:
    df[col] = df[col].replace(missing_tokens, np.nan)

# Xóa các bản ghi trùng hoàn toàn
duplicates_removed = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

print("Số bản ghi trùng đã xóa:", duplicates_removed)
print("Kích thước sau làm sạch cơ bản:", df.shape)

display(df.head())

Số bản ghi trùng đã xóa: 0
Kích thước sau làm sạch cơ bản: (30000, 41)


,order_id,customer_id,order_date,year,month,day,day_of_week,quarter,customer_age,customer_gender,...,delivery_days,order_status,returned,review_rating,customer_lifetime_value,profit_margin_percent,profit_amount,season,holiday_season,high_value_order
0,615717,CUST007322,2023-01-01,2023,1,1,Sunday,1,32,Male,...,2,Delivered,No,4.4,2144.92,23.10,15.75,Winter,No,No
1,626919,CUST004717,2023-01-01,2023,1,1,Sunday,1,50,Male,...,9,Delivered,No,4.1,817.17,8.57,3.32,Winter,No,No
2,615781,CUST004415,2023-01-01,2023,1,1,Sunday,1,61,Male,...,2,Delivered,No,5.0,541.16,29.72,45.67,Winter,No,No
3,621747,CUST004114,2023-01-01,2023,1,1,Sunday,1,34,Male,...,6,Returned,Yes,3.4,700.49,25.22,75.25,Winter,No,No
4,625881,CUST000145,2023-01-01,2023,1,1,Sunday,1,37,Male,...,8,Delivered,No,3.6,2133.77,24.64,26.06,Winter,No,No


Bước 5: Chuẩn hóa kiểu dữ liệu
## 5. Data Type Standardization

Xác định các cột ngày tháng và số liệu dựa trên tên cột.
Chuyển đổi dữ liệu khi phù hợp và kiểm tra các giá trị không thể chuyển đổi.

In [5]:
# Nhận diện cột ngày tháng dựa trên tên
date_keywords = ("date", "time", "timestamp")
date_columns = [
    col for col in df.columns
    if any(keyword in col for keyword in date_keywords)
]

for col in date_columns:
    converted = pd.to_datetime(df[col], errors="coerce")

    # Chỉ chuyển đổi khi có dữ liệu hợp lệ
    # và phần lớn giá trị không bị chuyển thành NaT
    original_non_null = df[col].notna().sum()

    if original_non_null > 0:
        success_rate = converted.notna().sum() / original_non_null

        if success_rate >= 0.8:
            df[col] = converted

# Nhận diện cột số dựa trên tên
numeric_keywords = (
    "price", "amount", "quantity", "qty",
    "total", "sales", "revenue", "discount",
    "tax", "cost", "profit"
)

for col in df.columns:
    if any(keyword in col for keyword in numeric_keywords):
        if df[col].dtype == "object":
            converted = pd.to_numeric(
                df[col].astype("string")
                .str.replace(",", "", regex=False)
                .str.replace("$", "", regex=False)
                .str.strip(),
                errors="coerce"
            )

            # Chỉ chuyển đổi nếu phần lớn giá trị hợp lệ
            original_non_null = df[col].notna().sum()

            if original_non_null > 0:
                success_rate = (
                    converted.notna().sum() / original_non_null
                )

                if success_rate >= 0.8:
                    df[col] = converted

print("Các cột ngày tháng:", date_columns)
print("Kiểu dữ liệu sau chuẩn hóa:")
display(df.dtypes.to_frame("Data Type"))

Các cột ngày tháng: ['order_date', 'customer_lifetime_value']
Kiểu dữ liệu sau chuẩn hóa:


,Data Type
order_id,int64
customer_id,object
order_date,datetime64[ns]
year,int64
month,int64
day,int64
day_of_week,object
quarter,int64
customer_age,int64
customer_gender,object


Bước 7: Xuất file CSV sạch và báo cáo
## 7. Export Clean Dataset

Xuất dữ liệu đã làm sạch, báo cáo giá trị thiếu và báo cáo chất lượng sau xử lý.
Các file đầu ra được sử dụng cho phân tích KPI và xây dựng dashboard.

In [6]:
# Báo cáo chất lượng sau làm sạch
quality_report = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing Percent": (df.isna().mean() * 100).round(2),
    "Unique Values": df.nunique(dropna=True),
    "Data Type": df.dtypes.astype(str)
})

# Xuất file
clean_file = OUTPUT_DIR / "ecommerce_orders_cleaned.csv"
quality_file = OUTPUT_DIR / "data_quality_report.csv"

df.to_csv(clean_file, index=False, encoding="utf-8-sig")
quality_report.to_csv(
    quality_file, encoding="utf-8-sig"
)

print("Hoàn thành quy trình làm sạch dữ liệu!")
print("Số dòng ban đầu:", len(df_raw))
print("Số dòng sau làm sạch:", len(df))
print("Số dòng đã loại bỏ:", len(df_raw) - len(df))
print("Số cột:", len(df.columns))
print("Số giá trị thiếu còn lại:", int(df.isna().sum().sum()))
print("File dữ liệu sạch:", clean_file)
print("File báo cáo chất lượng:", quality_file)

display(quality_report)

Hoàn thành quy trình làm sạch dữ liệu!
Số dòng ban đầu: 30000
Số dòng sau làm sạch: 30000
Số dòng đã loại bỏ: 0
Số cột: 41
Số giá trị thiếu còn lại: 0
File dữ liệu sạch: E:\FPT Polytechnic\DATN1\Data\output\ecommerce_orders_cleaned.csv
File báo cáo chất lượng: E:\FPT Polytechnic\DATN1\Data\output\data_quality_report.csv


,Missing Count,Missing Percent,Unique Values,Data Type
order_id,0,0.0,30000,int64
customer_id,0,0.0,8683,object
order_date,0,0.0,1461,datetime64[ns]
year,0,0.0,4,int64
month,0,0.0,12,int64
day,0,0.0,31,int64
day_of_week,0,0.0,7,object
quarter,0,0.0,4,int64
customer_age,0,0.0,58,int64
customer_gender,0,0.0,3,object
